# Etapa 1 — quantos falantes distintos há no corpus

Executa `pipeline_coleta_piloto/verificar_reincidencia.py` sobre os 52 arquivos já
coletados. O documento que governa esta etapa é `docs/plano_corpus/01-verificar-falantes.md`,
e ele deve ser lido antes: o que segue é apenas a execução.

## O que este notebook responde, e o que não responde

A diarização rotula locutores **dentro** de cada arquivo. Os rótulos não se conectam
entre arquivos, e o repórter que aparece em cinco episódios do mesmo canal é contado
como cinco pessoas. O notebook compara as vozes entre arquivos do mesmo estado e devolve
os **pares candidatos** a mesma pessoa.

Ele não decide. Fundir dois rótulos por engano apaga uma pessoa real do corpus; deixar
de fundir dois rótulos da mesma pessoa viola o teto de 5% por falante sem que ninguém
perceba. A decisão é da conferência humana, na seção 7, que este notebook instrumenta
tocando os dois trechos de cada par.

## Antes de executar

1. **GPU:** *Ambiente de execução* → *Alterar o tipo de ambiente* → **T4 GPU**. A troca
   reinicia o ambiente, então faça antes de tudo.
2. **Credencial:** `pyannote/embedding` é modelo de acesso condicionado. É preciso
   aceitar os termos em `https://huggingface.co/pyannote/embedding`, no botão *Agree and
   access repository* — sem isso o token é válido e o download é recusado, com erro que
   não menciona a causa. O token vai no painel de segredos do Colab, com o nome
   `HF_TOKEN`, nunca colado em célula.
3. **Material no Drive:** as pastas `audio/` e `registros_anonimizados/`.

## 1. Verificação do ambiente

In [ ]:
import shutil

def checar(nome, condicao, detalhe=""):
    print(f"{'OK   ' if condicao else 'FALHA'}  {nome}  {detalhe}")
    return condicao

ok = True
try:
    import torch
    ok &= checar("GPU", torch.cuda.is_available(),
                 torch.cuda.get_device_name(0) if torch.cuda.is_available()
                 else "sem GPU - ative em Ambiente de execucao")
except ImportError:
    print("torch ainda nao instalado; rode a proxima celula e volte aqui")
    ok = False

ok &= checar("ffmpeg", shutil.which("ffmpeg") is not None)

try:
    import numpy, numba, soundfile
    import pyannote.audio
    ok &= checar("pyannote.audio", True, pyannote.audio.__version__)
    ok &= checar("soundfile", True, soundfile.__version__)
    ok &= checar("numpy < 2.3", tuple(int(x) for x in numpy.__version__.split(".")[:2]) < (2, 3),
                 numpy.__version__)
except ImportError as e:
    print(f"dependencia ausente ({e.name}); rode a proxima celula")
    ok = False

print("\nambiente pronto" if ok else "\ncorrija os itens em FALHA antes de seguir")

## 2. Instalação

O limite de `numpy` é o mesmo do notebook do piloto, e pelo mesmo motivo: sem a
restrição, a instalação traz `numpy` mais novo do que o `numba` aceita, e o `numba`
entra por baixo do `pyannote`, via `librosa`.

In [ ]:
!pip install -q "numpy<2.3" "pyannote.audio>=3.1.0" "soundfile>=0.12.1"
!apt-get -qq install -y ffmpeg > /dev/null
print("instalado. Reinicie a sessao em Ambiente de execucao -> Reiniciar sessao,")
print("e execute novamente a celula de verificacao.")

## 3. Repositório, credencial e Drive

O `HF_TOKEN` aqui precisa dos termos de **`pyannote/embedding`** aceitos — modelo
distinto do usado na diarização, com aceitação própria. Ter rodado o notebook do piloto
não basta.

In [ ]:
# A correcao do comparador esta na branch abaixo. Enquanto ela nao for
# integrada a main, clonar a main traria a versao antiga do script -- que le de
# uma pasta vazia e relataria zero pares, sem erro.
BRANCH = "etapa1-verificar-falantes"

# Apagar antes de clonar torna a celula repetivel: sem isso, o segundo clone
# falha porque a pasta ja existe, e o notebook seguiria com a versao antiga
# do script sem que nada o indicasse.
!rm -rf /content/vies-nordeste-bertimbau
!git clone -q --branch {BRANCH} https://github.com/Aryazinha/vies-nordeste-bertimbau.git
!cd /content/vies-nordeste-bertimbau && git log --oneline -1

import os
from google.colab import drive, userdata

# O Drive e montado ANTES da leitura do token, e a leitura do token nao
# interrompe a celula. Na ordem inversa, um segredo ausente derrubava a celula
# antes do mount, e o sintoma aparecia so duas celulas adiante, como pasta de
# audio inexistente -- que faz suspeitar do upload, e nao da credencial.
drive.mount("/content/drive")
print("Drive montado:", os.path.isdir("/content/drive/MyDrive"))

try:
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
    print("HF_TOKEN carregado")
except Exception as e:
    print()
    print(f"HF_TOKEN NAO carregado ({type(e).__name__}).")
    print("Icone de chave na barra lateral -> Adicionar novo segredo -> nome HF_TOKEN,")
    print("com o acesso a este notebook LIGADO. Depois rode esta celula de novo.")
    print("So e necessario a partir da celula da comparacao; ate la pode seguir.")


## 4. Localização do material

Ajuste `PASTA_DRIVE` se você enviou a pasta para outro lugar. A célula confere que
áudio e registros existem e se correspondem: registro sem áudio sai da comparação em
silêncio, e um estado pode perder falantes sem que nada o indique.

In [ ]:
import json
from collections import Counter
from pathlib import Path

PASTA_DRIVE = Path("/content/drive/MyDrive/dataset_raw")   # ajuste se necessario

AUDIO = PASTA_DRIVE / "audio"
REGISTROS = PASTA_DRIVE / "registros_anonimizados"

assert AUDIO.is_dir(), f"pasta de audio nao encontrada: {AUDIO}"
assert REGISTROS.is_dir(), f"pasta de registros nao encontrada: {REGISTROS}"

registros = [json.loads(p.read_text(encoding="utf-8")) for p in sorted(REGISTROS.glob("*.json"))]
wavs = {p.name for p in AUDIO.glob("*.wav")}

sem_audio = [r["id"] for r in registros if r["arquivo"] not in wavs]
sem_diarizacao = [r["id"] for r in registros if not r.get("diarizacao")]

print(f"{len(registros)} registro(s), {len(wavs)} arquivo(s) .wav")
print("por estado:", dict(sorted(Counter(r["estado_alvo"] for r in registros).items())))
print("registros sem audio:", sem_audio or "nenhum")
print("registros sem diarizacao:", sem_diarizacao or "nenhum")

assert not sem_audio, "ha registro sem audio: a comparacao ficaria incompleta"
assert not sem_diarizacao, "ha registro sem diarizacao"

## 5. Comparação das vozes

Uma única carga do modelo para os seis estados. A comparação é interna a cada estado:
um falante do Recife e outro de São Paulo não disputam o mesmo teto.

Os resultados vão para o Drive, e não apenas para o disco da sessão, porque a
conferência humana da seção 7 costuma exigir mais de uma sessão.

In [ ]:
import sys
sys.path.insert(0, "/content/vies-nordeste-bertimbau/pipeline_coleta_piloto")

import importlib
import verificar_reincidencia as vr

# Recarrega o modulo: se a secao 3 tiver trazido uma correcao do script, o
# Python devolveria a versao ja em memoria, e a correcao nao valeria.
importlib.reload(vr)
from config import ESTADOS_VALIDOS

SAIDA = PASTA_DRIVE / "diarizacao"
SAIDA.mkdir(parents=True, exist_ok=True)

# A conferencia da secao 7 costuma exceder uma sessao do Colab. Sem reuso, cada
# retomada refaria na GPU uma comparacao que ja esta gravada no Drive. Ponha
# RECALCULAR = True para forcar a passagem de GPU, por exemplo ao mudar um
# parametro ou depois de acrescentar arquivos ao corpus na etapa 2.
RECALCULAR = False

prontos = {uf: SAIDA / f"reincidencia_{uf}.json" for uf in ESTADOS_VALIDOS}
reaproveitar = not RECALCULAR and all(c.exists() for c in prontos.values())

relatorios = {}
if reaproveitar:
    for uf, caminho in prontos.items():
        relatorios[uf] = json.loads(caminho.read_text(encoding="utf-8"))
    print("resultados lidos do Drive, sem nova passagem de GPU:")
    print("gerado em", relatorios[ESTADOS_VALIDOS[0]]["gerado_em"])
else:
    inferencia = vr._get_modelo_embedding()
    for uf in ESTADOS_VALIDOS:
        relatorio = vr.comparar_estado(uf, AUDIO, REGISTROS, inferencia,
                                       vr.LIMIAR_SIMILARIDADE, vr.LIMIAR_REGISTRO,
                                       vr.DURACAO_MINIMA_S, vr.DURACAO_ALVO_S)
        prontos[uf].write_text(json.dumps(relatorio, ensure_ascii=False, indent=2),
                               encoding="utf-8")
        relatorios[uf] = relatorio

for uf, rel in relatorios.items():
    print(uf, rel["resumo"])


## 6. O teto, antes da conferência

Os números abaixo são **teto, não resultado**: contam rótulos, e a conferência humana
só pode reduzi-los, ao fundir os que forem a mesma pessoa. A coluna dos rótulos sem
embedding é a medida da ignorância residual — vozes com menos de 8 s de fala, que não
são falantes verificados nem descartados.

In [ ]:
PISO = 20   # falantes distintos por estado (docs/fontes_coleta.md, 2.4.5)

print(f"{'UF':4} {'arq':>4} {'teto':>5} {'sem emb':>8} {'candidatos':>11} {'folga':>6}")
for uf, rel in relatorios.items():
    r = rel["resumo"]
    print(f"{uf:4} {r['arquivos']:>4} {r['teto_de_falantes']:>5} {r['rotulos_sem_embedding']:>8} "
          f"{r['pares_acima_do_limiar']:>11} {r['teto_de_falantes'] - PISO:>+6}")

total = sum(r["resumo"]["teto_de_falantes"] for r in relatorios.values())
print(f"\ntotal de rotulos com embedding: {total}")
print("Estes valores ainda nao descontam fusoes. A contagem final sai da secao 8.")

## 7. Conferência humana

Aqui está a decisão. Para cada par candidato, ouça os dois trechos e responda se é a
mesma pessoa.

**Ordem sugerida:** começar pela similaridade mais alta, que deve trazer as fusões
óbvias, e descer até que os pares deixem de ser plausíveis. **O ponto em que isso ocorre
é a calibração empírica do limiar**, e vale mais que o valor default de 0,75 — anote-o.

Por isso a saída registra também os pares abaixo do limiar, a partir de 0,50: sem eles,
descer a lista exigiria nova passagem de GPU.

Pares dentro do **mesmo canal** são os candidatos mais prováveis, porque o padrão
esperado de reincidência é o apresentador ou repórter do próprio canal.

In [ ]:
import torch
from IPython.display import Audio, Markdown, display

def _amostra(arquivo_id, segmentos, limite_s=12.0):
    """Concatena ate `limite_s` do que o comparador de fato ouviu naquele rotulo."""
    caminho = AUDIO / f"{arquivo_id}.wav"
    pedacos, acumulado = [], 0.0
    for inicio, fim in segmentos:
        duracao = min(fim - inicio, limite_s - acumulado)
        if duracao <= 0:
            break
        # Mesma leitura usada na comparacao: `soundfile`, e nao `torchaudio`,
        # que removeu `load` e `info` a partir da serie 2.9.
        onda, taxa = vr.ler_audio(caminho, inicio, duracao)
        pedacos.append(onda)
        acumulado += duracao
    return torch.cat(pedacos, dim=1).numpy(), taxa

def ouvir(uf, indice):
    par = relatorios[uf]["pares"][indice]
    marca = "MESMO CANAL" if par["mesmo_canal"] else "canais distintos"
    display(Markdown(
        f"### {uf} #{indice} - similaridade {par['similaridade']:.3f} "
        f"({'acima' if par['acima_do_limiar'] else 'abaixo'} do limiar) - {marca}\n"
        f"- **A** `{par['arquivo_a']}` / `{par['locutor_a']}` - {par['canal_a']}\n"
        f"- **B** `{par['arquivo_b']}` / `{par['locutor_b']}` - {par['canal_b']}"))
    for lado in ("a", "b"):
        onda, taxa = _amostra(par[f"arquivo_{lado}"], par[f"segmentos_{lado}"])
        display(Audio(onda, rate=taxa))

def faixas(passo=0.05):
    """
    Quantos pares ha em cada faixa de similaridade, por estado, e quantas fusoes
    cada estado suporta antes de cair abaixo do piso.

    A margem e o que decide ate onde descer na escuta. Um estado com folga de +1
    precisa que a lista seja varrida bem abaixo do limiar, porque um unico par
    perdido o derruba; um com folga de +10 nao muda de conclusao por causa de
    pares duvidosos.
    """
    limites = [0.90, 0.85, 0.80, 0.75, 0.70, 0.65, 0.60, 0.55, 0.50]
    print(f"{'UF':4} {'margem':>7} " + " ".join(f">{x:.2f}" for x in limites))
    for uf, rel in relatorios.items():
        sims = [par["similaridade"] for par in rel["pares"]]
        margem = rel["resumo"]["teto_de_falantes"] - PISO
        contagens = [sum(1 for s in sims if s >= x) for x in limites]
        print(f"{uf:4} {margem:>+7} " + " ".join(f"{c:>5}" for c in contagens))
    print()
    print("As colunas sao cumulativas: '>0.70' inclui os pares de '>0.75'.")
    print("margem = quantas fusoes o estado suporta antes de cair abaixo de", PISO)

faixas()

ouvir("PB", 0)   # troque o estado e o indice a cada par conferido


### 7.1 Registro dos vereditos

`marcar` grava no Drive a cada chamada, de modo que a conferência sobreviva ao
encerramento da sessão. Uma etapa concluída cujo resultado só existe no histórico da
sessão está perdida, e o projeto já perdeu material assim uma vez.

In [ ]:
VEREDITOS = SAIDA / "vereditos_reincidencia.json"
vereditos = json.loads(VEREDITOS.read_text(encoding="utf-8")) if VEREDITOS.exists() else {}

def marcar(uf, indice, mesma_pessoa, nota=""):
    par = relatorios[uf]["pares"][indice]
    chave = f"{uf}|{par['arquivo_a']}|{par['locutor_a']}|{par['arquivo_b']}|{par['locutor_b']}"
    # O veredito guarda os proprios rotulos, e nao so o indice: o indice muda se a
    # comparacao for repetida com outro limiar, e a conferencia se desalinharia em
    # silencio, apontando para pares diferentes dos que foram ouvidos.
    vereditos[chave] = {
        "estado": uf, "indice": indice, "similaridade": par["similaridade"],
        "rotulo_a": [par["arquivo_a"], par["locutor_a"]],
        "rotulo_b": [par["arquivo_b"], par["locutor_b"]],
        "veredito": "mesma_pessoa" if mesma_pessoa else "pessoas_distintas",
        "mesmo_canal": par["mesmo_canal"], "nota": nota,
    }
    VEREDITOS.write_text(json.dumps(vereditos, ensure_ascii=False, indent=2), encoding="utf-8")
    print(f"{chave} -> {vereditos[chave]['veredito']}  ({len(vereditos)} par(es) conferido(s))")

def pendentes(uf):
    """Pares do estado ainda sem veredito, do mais similar ao menos."""
    conferidos = {v["indice"] for v in vereditos.values() if v["estado"] == uf}
    return [i for i in range(len(relatorios[uf]["pares"])) if i not in conferidos]

# marcar("PB", 0, True, "mesma reporter, dois episodios")

## 8. Apuração

O número de falantes distintos **não** é o de rótulos menos o de pares confirmados: se
os rótulos A e B são a mesma pessoa, e B e C também, os três pares possíveis descrevem
uma pessoa só. A apuração agrupa os rótulos em componentes conexos, o que dá a contagem
correta qualquer que seja a quantidade de pares confirmados sobre a mesma pessoa.

In [ ]:
def apurar(uf):
    rotulos = [(r["arquivo"], r["locutor"]) for r in relatorios[uf]["rotulos"]]
    pai = {r: r for r in rotulos}

    def raiz(x):
        while pai[x] != x:
            pai[x] = pai[pai[x]]
            x = pai[x]
        return x

    fusoes, orfaos = 0, []
    for v in vereditos.values():
        if v["estado"] != uf or v["veredito"] != "mesma_pessoa":
            continue
        rotulo_a, rotulo_b = tuple(v["rotulo_a"]), tuple(v["rotulo_b"])
        if rotulo_a not in pai or rotulo_b not in pai:
            orfaos.append(v)      # rotulo sumiu entre execucoes; nao silenciar
            continue
        a, b = raiz(rotulo_a), raiz(rotulo_b)
        if a != b:
            pai[a] = b
            fusoes += 1

    return {"rotulos": len(rotulos), "fusoes": fusoes,
            "falantes_distintos": len({raiz(r) for r in rotulos}),
            "sem_embedding": relatorios[uf]["resumo"]["rotulos_sem_embedding"],
            "vereditos_orfaos": len(orfaos)}

PISO = 20
print(f"{'UF':4} {'rotulos':>8} {'fusoes':>7} {'distintos':>10} {'piso':>5} {'situacao':>10} {'sem emb':>8}")
apuracao = {}
for uf in relatorios:
    a = apurar(uf)
    apuracao[uf] = a
    situacao = "ok" if a["falantes_distintos"] >= PISO else f"faltam {PISO - a['falantes_distintos']}"
    print(f"{uf:4} {a['rotulos']:>8} {a['fusoes']:>7} {a['falantes_distintos']:>10} "
          f"{PISO:>5} {situacao:>10} {a['sem_embedding']:>8}")
    if a["vereditos_orfaos"]:
        print(f"     ATENCAO: {a['vereditos_orfaos']} veredito(s) sem rotulo correspondente")

(SAIDA / "apuracao_falantes.json").write_text(
    json.dumps(apuracao, ensure_ascii=False, indent=2), encoding="utf-8")
print("\ngravado em", SAIDA / "apuracao_falantes.json")

## 9. Ao terminar

Baixe do Drive `reincidencia_*.json`, `vereditos_reincidencia.json` e
`apuracao_falantes.json`, e então, conforme a seção 8 de
`docs/plano_corpus/01-verificar-falantes.md`:

1. Registrar em `01-verificar-falantes.md` os números apurados, **o limiar calibrado** e
   a data.
2. Atualizar `docs/dataset-spec.md`, na seção "Camada de execução, em números", que
   ainda afirma que a verificação nunca rodou.
3. Atualizar `docs/pendencias.md`, seção 6.4, e o item #5 do registro de pendentes.
4. Havendo déficit, anotá-lo por estado em `02-completar-coleta.md`, que é o insumo
   daquela etapa.

Registrar também **os rótulos sem embedding**: eles não são falantes verificados nem
descartados, e o seu número limita o que se pode afirmar sobre a contagem.